# 03 - Paraphrase COTs

Generate all COT variants using Qwen3-8B as paraphraser:
- **paraphrase_light**: Reword while preserving all steps and numbers
- **paraphrase_heavy**: Compress to just key steps and intermediate results

Also generate Python-based transformations:
- **shuffled_steps**: Randomly reorder COT steps
- **corrupted_numbers**: Replace intermediate numbers with random values

In [1]:
import subprocess, sys
from pathlib import Path

WORKSPACE = Path("/workspace/11-4-2026")
REPO_DIR = WORKSPACE / "legibility"

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/JackHopkins/legibility.git", str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
from lib.config import *

for d in [CACHE_DIR, COT_CACHE, PARAPHRASE_CACHE, PREFILL_CACHE, RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Already up to date.


In [ ]:
import json
import re
from tqdm import tqdm
from lib.paraphrase import shuffle_steps, corrupt_numbers
from lib.prompts import build_paraphrase_light_messages, build_paraphrase_heavy_messages


def strip_think_tags(text: str) -> str:
    """Remove <think>...</think> blocks from model output."""
    cleaned = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    # Also handle unclosed think tags (model hit max tokens mid-think)
    cleaned = re.sub(r"<think>.*", "", cleaned, flags=re.DOTALL).strip()
    return cleaned

## Load Original COTs

In [3]:
cots = {}
for p in sorted(COT_CACHE.glob("*.json"), key=lambda x: int(x.stem)):
    data = json.loads(p.read_text())
    cots[data["problem_id"]] = data

print(f"Loaded {len(cots)} COTs")
# Filter to only problems with valid COTs
valid_cots = {pid: c for pid, c in cots.items() if c["cot_text"] and c["error"] is None}
print(f"Valid COTs (non-empty, no error): {len(valid_cots)}")

Loaded 1319 COTs
Valid COTs (non-empty, no error): 1319


## Load Paraphraser Model (Qwen3-8B)

In [4]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

llm = LLM(
    model=PARAPHRASER_MODEL,
    dtype="bfloat16",
    max_model_len=4096,
)
tokenizer = AutoTokenizer.from_pretrained(PARAPHRASER_MODEL)
sampling = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_COT_TOKENS)
print(f"Loaded paraphraser: {PARAPHRASER_MODEL}")

INFO 04-11 12:33:56 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'max_model_len': 4096, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-8B'}
INFO 04-11 12:33:56 [model.py:533] Resolved architecture: Qwen3ForCausalLM
INFO 04-11 12:33:56 [model.py:1582] Using max model len 4096
INFO 04-11 12:33:56 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 04-11 12:33:56 [vllm.py:754] Asynchronous scheduling is enabled.
(EngineCore pid=74516) INFO 04-11 12:33:57 [core.py:103] Initializing a V1 LLM engine (v0.18.0) with config: model='Qwen/Qwen3-8B', speculative_config=None, tokenizer='Qwen/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantiz

(EngineCore pid=74516) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=74516) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.


Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


(EngineCore pid=74516) INFO 04-11 12:34:19 [default_loader.py:384] Loading weights took 19.10 seconds
(EngineCore pid=74516) INFO 04-11 12:34:20 [gpu_model_runner.py:4566] Model loading took 15.27 GiB memory and 20.034663 seconds
(EngineCore pid=74516) INFO 04-11 12:34:26 [backends.py:988] Using cache directory: /root/.cache/vllm/torch_compile_cache/8fc724e697/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=74516) INFO 04-11 12:34:26 [backends.py:1048] Dynamo bytecode transform time: 5.76 s
(EngineCore pid=74516) INFO 04-11 12:34:27 [backends.py:371] Cache the graph of compile range (1, 16384) for later use
(EngineCore pid=74516) INFO 04-11 12:34:31 [backends.py:387] Compiling a graph for compile range (1, 16384) takes 4.81 s
(EngineCore pid=74516) INFO 04-11 12:34:33 [decorators.py:627] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/cc5a8a7ad433ab700b614c0db8aacedd8a663e83e6a5382b8d833c2d8294081a/rank_0_0/model
(EngineCore pid=74516) 

(EngineCore pid=74516) 2026-04-11 12:34:35,195 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=74516) 2026-04-11 12:34:35,206 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 26.18it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:01<00:00, 33.35it/s]


(EngineCore pid=74516) INFO 04-11 12:34:39 [gpu_model_runner.py:5746] Graph capturing finished in 4 secs, took 0.97 GiB
(EngineCore pid=74516) INFO 04-11 12:34:39 [gpu_worker.py:617] CUDA graph pool memory: 0.97 GiB (actual), 0.84 GiB (estimated), difference: 0.12 GiB (12.9%).
(EngineCore pid=74516) INFO 04-11 12:34:39 [core.py:281] init engine (profile, create kv cache, warmup model) took 19.20 seconds
INFO 04-11 12:34:40 [llm.py:391] Supported tasks: ['generate']
Loaded paraphraser: Qwen/Qwen3-8B


## Generate Light Paraphrases

In [ ]:
condition = "paraphrase_light"
done_ids = {
    int(p.stem.split("_")[-1])
    for p in PARAPHRASE_CACHE.glob(f"{condition}_*.json")
}
todo = [(pid, c) for pid, c in valid_cots.items() if pid not in done_ids]
print(f"[{condition}] {len(done_ids)} done, {len(todo)} remaining")

CHUNK_SIZE = 32

for i in tqdm(range(0, len(todo), CHUNK_SIZE), desc=condition):
    chunk = todo[i : i + CHUNK_SIZE]

    prompts = []
    for pid, cot_data in chunk:
        messages = build_paraphrase_light_messages(cot_data["cot_text"])
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
        prompts.append(prompt)

    outputs = llm.generate(prompts, sampling)

    for (pid, cot_data), output in zip(chunk, outputs):
        paraphrased = strip_think_tags(output.outputs[0].text)

        result = {
            "problem_id": pid,
            "condition": condition,
            "original_cot": cot_data["cot_text"],
            "paraphrased_cot": paraphrased,
        }
        (PARAPHRASE_CACHE / f"{condition}_{pid}.json").write_text(json.dumps(result))

print(f"Light paraphrasing complete.")

## Generate Heavy Paraphrases

In [ ]:
condition = "paraphrase_heavy"
done_ids = {
    int(p.stem.split("_")[-1])
    for p in PARAPHRASE_CACHE.glob(f"{condition}_*.json")
}
todo = [(pid, c) for pid, c in valid_cots.items() if pid not in done_ids]
print(f"[{condition}] {len(done_ids)} done, {len(todo)} remaining")

for i in tqdm(range(0, len(todo), CHUNK_SIZE), desc=condition):
    chunk = todo[i : i + CHUNK_SIZE]

    prompts = []
    for pid, cot_data in chunk:
        messages = build_paraphrase_heavy_messages(cot_data["cot_text"])
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
        prompts.append(prompt)

    outputs = llm.generate(prompts, sampling)

    for (pid, cot_data), output in zip(chunk, outputs):
        paraphrased = strip_think_tags(output.outputs[0].text)

        result = {
            "problem_id": pid,
            "condition": condition,
            "original_cot": cot_data["cot_text"],
            "paraphrased_cot": paraphrased,
        }
        (PARAPHRASE_CACHE / f"{condition}_{pid}.json").write_text(json.dumps(result))

print(f"Heavy paraphrasing complete.")

In [7]:
# Free GPU for model swap in next notebook
del llm
import gc; gc.collect()
import torch; torch.cuda.empty_cache()
print("Paraphraser model freed.")

Paraphraser model freed.


## Generate Shuffled Steps (Python, no LLM)

In [8]:
condition = "shuffled_steps"
done_ids = {
    int(p.stem.split("_")[-1])
    for p in PARAPHRASE_CACHE.glob(f"{condition}_*.json")
}
todo = [(pid, c) for pid, c in valid_cots.items() if pid not in done_ids]
print(f"[{condition}] {len(done_ids)} done, {len(todo)} remaining")

for pid, cot_data in tqdm(todo, desc=condition):
    shuffled = shuffle_steps(cot_data["cot_text"], seed=pid)

    result = {
        "problem_id": pid,
        "condition": condition,
        "original_cot": cot_data["cot_text"],
        "paraphrased_cot": shuffled,
    }
    (PARAPHRASE_CACHE / f"{condition}_{pid}.json").write_text(json.dumps(result))

print(f"Shuffled steps complete.")

[shuffled_steps] 0 done, 1319 remaining


shuffled_steps: 100%|██████████| 1319/1319 [00:07<00:00, 188.10it/s]

Shuffled steps complete.


## Generate Corrupted Numbers (Python, no LLM)

In [9]:
condition = "corrupted_numbers"
done_ids = {
    int(p.stem.split("_")[-1])
    for p in PARAPHRASE_CACHE.glob(f"{condition}_*.json")
}
todo = [(pid, c) for pid, c in valid_cots.items() if pid not in done_ids]
print(f"[{condition}] {len(done_ids)} done, {len(todo)} remaining")

for pid, cot_data in tqdm(todo, desc=condition):
    corrupted = corrupt_numbers(
        cot_data["cot_text"],
        final_answer=cot_data["gold_answer"],
        seed=pid,
    )

    result = {
        "problem_id": pid,
        "condition": condition,
        "original_cot": cot_data["cot_text"],
        "paraphrased_cot": corrupted,
    }
    (PARAPHRASE_CACHE / f"{condition}_{pid}.json").write_text(json.dumps(result))

print(f"Corrupted numbers complete.")

[corrupted_numbers] 0 done, 1319 remaining


corrupted_numbers: 100%|██████████| 1319/1319 [00:07<00:00, 173.70it/s]

Corrupted numbers complete.


## Verify Paraphrase Quality

In [10]:
# Show sample paraphrases for manual inspection
sample_pid = list(valid_cots.keys())[0]

print("=" * 80)
print(f"Problem {sample_pid}")
print("=" * 80)

print("\n--- ORIGINAL COT ---")
print(valid_cots[sample_pid]["cot_text"][:500])

for cond in ["paraphrase_light", "paraphrase_heavy", "shuffled_steps", "corrupted_numbers"]:
    path = PARAPHRASE_CACHE / f"{cond}_{sample_pid}.json"
    if path.exists():
        data = json.loads(path.read_text())
        print(f"\n--- {cond.upper()} ---")
        print(data["paraphrased_cot"][:500])

Problem 0

--- ORIGINAL COT ---
Okay, let's see. Natalia sold clips to 48 friends in April. Then in May, she sold half as many as in April. So I need to find out how many she sold in May first. Half of 48 would be 24, right? Because 48 divided by 2 is 24. So in May, she sold 24 clips.

Now, to find the total number of clips sold in both months, I need to add the number from April and May together. That would be 48 (April) plus 24 (May). Let me do the addition: 48 + 24. Hmm, 40 + 20 is 60, and 8 + 4 is 12. So 60 + 12 is 72. So 

--- PARAPHRASE_LIGHT ---
<think>
Okay, let's see. Natalia sold clips to 48 friends in April. Then in May, she sold half as many as in April. So I need to find out how many she sold in May first. Half of 48 would be 24, right? Because 48 divided by 2 is 24. So in May, she sold 24 clips.

Now, to find the total number of clips sold in both months, I need to add the number from April and May together. That would be 48 (April) plus 24 (May). Let me do the addition: 

In [11]:
# Count paraphrases per condition
for cond in ["paraphrase_light", "paraphrase_heavy", "shuffled_steps", "corrupted_numbers"]:
    count = len(list(PARAPHRASE_CACHE.glob(f"{cond}_*.json")))
    print(f"{cond}: {count} paraphrases")

paraphrase_light: 1319 paraphrases
paraphrase_heavy: 1319 paraphrases
shuffled_steps: 1319 paraphrases
corrupted_numbers: 1319 paraphrases
